# Estudo de caso 2.5 - O efeito do porte de armas nas taxas de homicídio


Configuração do *notebook*:

Para isso, siga o link que aparece na saída da seguinte célula uma vez executada. Copie o código que aparece na tela e insira-o na saída da célula. Assim que visualizar a mensagem: `Google Drive sincronizado com sucesso!` poderá continuar executando o restante das células.

In [1]:
! pip install rpy2==3.5.1

     ---------------------------------------- 0.0/201.7 kB ? eta -:--:--
     --------------------- ---------------- 112.6/201.7 kB 3.2 MB/s eta 0:00:01
     -------------------------------------- 201.7/201.7 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for rpy2: filename=rpy2-3.5.1-py3-none-any.whl size=205903 sha256=ae8c8fabb1df7105e9140e374637b2bacdf4cc590d0973e6c1903de988e8da92
  Stored in directory: c:\users\andre\appdata\local\pip\cache\wheels\e9\55\d1\47be85a5f3f1e1f4d1e91cb5e3a4dcb40dd72147f184c5a5ef
Successfully built rpy2


In [2]:
from google.colab import auth
auth.authenticate_user()

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
data_drop = drive.CreateFile({'id':'1wRXji7OJE6mAs_obWZc0U_fHyW63JP82'})
data_drop.GetContentFile('gun_clean.csv')
data_drop = drive.CreateFile({'id':'1aIbjCQWQXHaCTWku1asywKeNq6dTTC9e'})
data_drop.GetContentFile('Cond-comp.R')
data_drop = drive.CreateFile({'id':'1IQqdJnEiSqXSfOgpRxozyxlthDYiW1bX'})
data_drop.GetContentFile('Functions.R')

print('Google Drive sincronizado com sucesso!')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
%load_ext rpy2.ipython

Instalar e importar bibliotecas:

(ignorar resultados a não ser que não apareça a frase: `Bibliotecas instaladas com sucesso!`)

(pode levar alguns minutos)

In [ ]:
%%R

install.packages("foreign");
install.packages("quantreg");
install.packages("mnormt");
install.packages("gbm");
install.packages("glmnet");
install.packages("MASS");
install.packages("rpart");
install.packages("sandwich");
install.packages("hdm");
install.packages("randomForest");
install.packages("nnet")
install.packages("neuralnet")
install.packages("caret")
install.packages("matrixStats")
install.packages("devtools")
install.packages("plyr")

cat('\nBibliotecas instaladas com sucesso!')

(as ‘lib’ is unspecified)







	‘/tmp/Rtmp7huwLx/downloaded_packages’

(as ‘lib’ is unspecified)



















	‘/tmp/Rtmp7huwLx/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/Rtmp7huwLx/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/Rtmp7huwLx/downloaded_packages’

(as ‘lib’ is unspecified)


































	‘/tmp/Rtmp7huwLx/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/Rtmp7huwLx/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/Rtmp7huwLx/downloaded_packages’

(as ‘lib’ is unspecified)














	‘/tmp/Rtmp7huwLx/downloaded_packages’

(as ‘lib’ is unspecified)



















	‘/tmp/Rtmp7huwLx/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/Rtmp7huwLx/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/Rtmp7huwLx/downloaded_packages’

(as ‘lib’ is unspecified)














	‘/tmp/Rtmp7huwLx/downloaded_packages’

(as ‘lib’ is unspecified)




































Bibliotecas instaladas com sucesso!

Importar bibliotecas:

In [ ]:
%%R

library(foreign);
library(quantreg);
library(mnormt);
library(gbm);
library(glmnet);
library(MASS);
library(rpart);
library(sandwich);
library(hdm);
library(randomForest);
library(nnet)
library(neuralnet)
library(caret)
library(matrixStats)
library(devtools)
library(plyr)

cat('\nBibliotecas importadas com sucesso!')


Attaching package: ‘SparseM’



    backsolve








Attaching package: ‘ggplot2’



    margin




Attaching package: ‘plyr’



    count





Bibliotecas importadas com sucesso!

Importar funções de arquivos externos:

In [ ]:
%%R
source("Cond-comp.R")
source("Functions.R")

## Dados

In [ ]:
%%R
# Estabelecer gerador de números aleatórios
set.seed(1)

# Carregar o banco de dados
data <- read.csv("gun_clean.csv", sep = ",", header = TRUE)

# Tabela para salvar os resultados
table <- matrix(0,12,2)

Recuperar os nomes das variáveis do banco de dados:

In [ ]:
%%R
varlist <- function (df=NULL,type=c("numeric","factor","character"), pattern="", exclude=NULL) {
  vars <- character(0)
  if (any(type %in% "numeric")) {
    vars <- c(vars,names(df)[sapply(df,is.numeric)])
  }
  if (any(type %in% "factor")) {
    vars <- c(vars,names(df)[sapply(df,is.factor)])
  }
  if (any(type %in% "character")) {
    vars <- c(vars,names(df)[sapply(df,is.character)])
  }
  vars[(!vars %in% exclude) & grepl(vars,pattern=pattern)]
}

Criar variáveis:

In [ ]:
%%R
# Variáveis binárias para os efeitos temporais e de municípios
fixed  <- grep("X_Jfips", names(data), value=TRUE, fixed=TRUE)
year   <- varlist(data, pattern="X_Tyear")

# Variáveis de controle do censo
census     <- NULL
census_var <- c("^AGE", "^BN", "^BP", "^BZ", "^ED", "^EL","^HI", "^HS", "^INC", "^LF", "^LN", "^PI", "^PO", "^PP", "^PV", "^SPR", "^VS")

for(i in 1:length(census_var)){
  census  <- append(census, varlist(data, pattern=census_var[i]))
}

# Regressor objetivo (variável de tratamento)
d     <- "logfssl"

# Variável de resultado
y     <- "logghomr"

# Outras variáveis de controle
X1    <- c("logrobr", "logburg", "burg_missing", "robrate_missing")
X2    <- c("newblack", "newfhh", "newmove", "newdens", "newmal")

## Metodologia

Particionamento dos efeitos temporais e de municípios:

In [ ]:
%%R
# Novo conjunto de dados para as variáveis particionadas
rdata    <- as.data.frame(data$CountyCode)
colnames(rdata) <- "CountyCode"

# Variáveis a serem particionadas
varlist <- c(y, d,X1, X2, census)

# Particionamento dos efeitos temporais e de município nas variáveis de varlist
for(i in 1:length(varlist)){
  form <- as.formula(paste(varlist[i], "~", paste(paste(year,collapse="+"),  paste(fixed,collapse="+"), sep="+")))
  rdata[, varlist[i]] <- lm(form, data)$residuals
}

Regressão linear:

In [ ]:
%%R
form1 <- as.formula(paste(y, "~", d ))
form2 <- as.formula(paste(y, "~", paste(d, paste(X1,collapse="+"), paste(X2,collapse="+"), paste(census,collapse="+"),   sep="+")))

table[1,1:2] <- summary(lm(form1, rdata))$coef[2,1:2]
table[2,1:2] <- summary(lm(form2, rdata))$coef[2,1:2]

Métodos Duplos de Aprendizagem de Máquina (*Double Machine Learning Methods*, Chernozhukov et al., 2018):

(pode levar alguns minutos para executar)

In [ ]:
%%R
# Variável de resultado
y.name      <-  y;

# Regressor objetivo (também: variável de tratamento)
d.name      <- d;

# Controles
x.name      <- paste(paste(X1,collapse="+"),  paste(X2,collapse="+"), paste(census,collapse="+"), sep="+") # use this for tree-based methods like forests and boosted trees

# Nomes dos métodos
method      <- c("RLasso","PostRLasso", "Forest", "Boosting", "Trees", "Lasso", "Ridge", "Elnet", "Nnet")

# Função que retorna a estimativa de coeficientes por Métodos Duplos de Aprendizagem de Máquina
# lista de argumentos:
#    1 : dados
#    2 : variável de resultado
#    3 : regressor objetivo
#    4 : variáveis de controle para métodos baseados em árvores
#    5 : variáveis de controle para modelos lineares (especificação flexível) - 4 e 5 são iguais neste exemplo
#    6 : nomes dos métodos
#    7 : número de separações de dados
#    8 : modelo particionado

res <- DoubleML(rdata, y.name, d.name, x.name, x.name, method=method, 2 ,"plinear")

table[3,1:2]   <-res[,1]
table[4,1:2]   <-res[,2]
table[5,1:2]  <-res[,3]
table[6,1:2] <-res[,4]
table[7,1:2] <-res[,5]
table[8,1:2] <-res[,6]
table[9,1:2] <-res[,7]
table[10,1:2] <-res[,8]
table[11,1:2] <-res[,9]
table[12,1:2] <-res[,10]

RLasso 
  split 1 
  split 2 
PostRLasso 
  split 1 
  split 2 
Forest 
  split 1 
  split 2 
Boosting 
  split 1 
  split 2 
Trees 
  split 1 
  split 2 
Lasso 
  split 1 
  split 2 
Ridge 
  split 1 
  split 2 
Elnet 
  split 1 
  split 2 
Nnet 
  split 1 
  split 2 
  best method for E[Y|X]: Boosting 
  best method for E[D|X]: RLasso 


## Resultados

In [ ]:
%%R
# Atribuir nomes de colunas e filas
colnames(table) <- c("Estimativa (beta)", "Erro padrão")
rownames(table) <- c("OLS (sem controles)", "OLS", "Lasso", "Pós-Lasso","Random Forest",
                     "Boosting Trees","Árvore podada","Lasso com VC", "Ridge com VC",
                     "Elastic Net com VC","Rede Neural", "Melhor preditor")

# Mostrar resultados
print(table, digit=3)

                    Estimativa (beta) Erro padrão
OLS (sem controles)             0.282      0.0573
OLS                             0.192      0.0565
Lasso                           0.238      0.0596
Pós-Lasso                       0.249      0.0589
Random Forest                   0.238      0.0896
Boosting Trees                  0.177      0.0681
Árvore podada                   0.173      0.0603
Lasso com VC                    0.188      0.0588
Ridge com VC                    0.196      0.0574
Elastic Net com VC              0.183      0.0590
Rede Neural                     0.289      0.0732
Melhor preditor                 0.245      0.0679
